# Tiny-LLM-From-Scratch (~50M) — Pre-Training & HuggingFace Publish Notebook

This notebook trains a **~50M parameter GPT-2-style decoder-only Transformer** completely from scratch
in PyTorch on **FineWeb-Edu** (educational web text), then publishes the trained model and custom
BPE tokenizer to the **Hugging Face Hub**.

### Target Architecture (~50M — optimised for T4 16 GB VRAM)
| Hyperparameter | Value |
|---|---|
| **Total parameters** | ~50,643,200 (~50.6 M) |
| **Vocabulary** | 16,384 Byte-Level BPE tokens |
| **Context length (T)** | 256 tokens |
| **Embedding dim (C)** | 640 |
| **Attention heads (H)** | 10 (head\_dim = 64) |
| **Transformer layers (N)** | 6 |
| **FFN hidden dim** | 2,560 (4 × C) |
| **Training data** | 20 M tokens (FineWeb-Edu sample-10BT) |
| **Effective batch size** | 32 × 4 = 128 sequences |
| **Max training steps** | 5,000 |
| **Hardware** | Google Colab T4 GPU (16 GB VRAM) + AMP FP16 |

### Parameter Breakdown
With C=640, H=10 (head_dim=64), N=6, V=16384, T=256, no weight-tying:
- Token embedding:   V × C = 16384 × 640 = **10,485,760**
- Position embedding: T × C = 256 × 640 = **163,840**
- Per Transformer block (Attn QKV + Out + MLP in/out + biases): **~4,920,320**
- 6 blocks: **~29,521,920**
- Final LayerNorm: **1,280**
- LM Head (no weight tying): V × C = **10,485,760**
- **Grand total: ~50,658,560**

> **Tip**: All cells must be run top-to-bottom in order. Skipping cells will cause errors.


## 1. Environment Setup & Dependencies

Install the few extra packages that Colab doesn't ship with and print GPU info.

In [1]:
# Install required dependencies (Colab already has torch, numpy, tqdm)
# tokenizers  -> HuggingFace fast BPE tokenizer library
# datasets    -> streaming FineWeb-Edu data
# huggingface_hub -> uploading the model to HF Hub
%pip install -q tokenizers datasets huggingface_hub

import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device      : {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM            : {props.total_memory / 1e9:.1f} GB")


PyTorch Version : 2.11.0+cu128
CUDA Available  : True
GPU Device      : Tesla T4
VRAM            : 15.6 GB


## 2. BPE Tokenizer — Implementation & Training on FineWeb-Edu

We build a **Byte-Level BPE tokenizer** (the same kind GPT-2 uses) and train it from scratch
on 80,000 documents streamed from FineWeb-Edu.

### Why Byte-Level BPE?
- Every Unicode character is first broken into raw bytes (0–255).
- Then frequent byte-sequences are merged together to form subword tokens.
- This means **every possible text can be encoded** — no unknown-word problem.

### Critical fix: `initial_alphabet=ByteLevelPreTokenizer.alphabet()`
Without this, bytes that never appear in the training corpus are missing from the vocabulary.
That causes round-trip decoding failures for characters like uppercase `T`, digits, etc.
Always pass `initial_alphabet` to guarantee all 256 bytes are in the vocab.

In [2]:
import os
import json
from pathlib import Path
from typing import List, Iterator, Optional

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel as ByteLevelPreTokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

# Special tokens that are always reserved in the vocabulary
ENDOFTEXT_TOKEN = "<|endoftext|>"   # appended at the end of every document
PAD_TOKEN       = "<|pad|>"          # used for padding (not needed during training)


class BPETokenizer:
    """
    Byte-Level BPE Tokenizer wrapper around the HuggingFace 'tokenizers' library.

    Usage
    -----
    tok = BPETokenizer(vocab_size=16384)
    tok.train(texts=my_text_iterator())
    ids = tok.encode("Hello world")   # [541, 995, ...]
    text = tok.decode(ids)             # "Hello world"
    tok.save("artifacts/tokenizer")
    """

    def __init__(self, vocab_size: int = 16_384):
        self.vocab_size = vocab_size
        self._tokenizer: Optional[Tokenizer] = None
        self.endoftext_id: Optional[int] = None
        self.pad_id: Optional[int] = None

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------
    def train(self, texts: Iterator[str], show_progress: bool = True) -> None:
        """Train the BPE tokenizer from a stream of raw text strings."""
        # 1. Create a new Tokenizer with an empty BPE model
        tokenizer = Tokenizer(BPE())

        # 2. Set byte-level pre-tokeniser: splits text into Unicode code points
        #    then maps each to a byte, so every character becomes 1+ byte tokens.
        tokenizer.pre_tokenizer = ByteLevelPreTokenizer(add_prefix_space=False)

        # 3. Set byte-level decoder: reverses the byte->string mapping on output.
        tokenizer.decoder = ByteLevelDecoder(add_prefix_space=False)

        # 4. Configure the BPE trainer
        trainer = BpeTrainer(
            vocab_size=self.vocab_size,
            special_tokens=[ENDOFTEXT_TOKEN, PAD_TOKEN],
            # KEY: seed the vocab with ALL 256 possible bytes so no byte is ever
            # missing even if it didn't appear in the training corpus.
            initial_alphabet=ByteLevelPreTokenizer.alphabet(),
            show_progress=show_progress,
            min_frequency=2,   # a byte-pair must appear at least twice to be merged
        )

        # 5. Run training
        tokenizer.train_from_iterator(texts, trainer=trainer)
        self._tokenizer = tokenizer
        self._cache_special_token_ids()

    # ------------------------------------------------------------------
    # Encode / Decode
    # ------------------------------------------------------------------
    def encode(self, text: str) -> List[int]:
        """Convert a string to a list of integer token IDs."""
        return self._tokenizer.encode(text).ids

    def decode(self, ids: List[int], skip_special_tokens: bool = True) -> str:
        """Convert a list of integer token IDs back to a string."""
        return self._tokenizer.decode(ids, skip_special_tokens=skip_special_tokens)

    # ------------------------------------------------------------------
    # Save / Load
    # ------------------------------------------------------------------
    def save(self, directory: str) -> None:
        """Save the tokenizer to disk (creates the directory if needed)."""
        save_dir = Path(directory)
        save_dir.mkdir(parents=True, exist_ok=True)
        self._tokenizer.model.save(str(save_dir))         # saves vocab.json + merges.txt
        self._tokenizer.save(str(save_dir / "full_tokenizer.json"))  # full state
        # Also save a small config so we can reload IDs without re-scanning
        config = {
            "vocab_size": self.vocab_size,
            "endoftext_id": self.endoftext_id,
            "pad_id": self.pad_id,
            "endoftext_token": ENDOFTEXT_TOKEN,
            "pad_token": PAD_TOKEN,
        }
        with open(save_dir / "tokenizer_config.json", "w") as f:
            json.dump(config, f, indent=2)

    @classmethod
    def load(cls, directory: str) -> "BPETokenizer":
        """Load a previously saved tokenizer from disk."""
        load_dir = Path(directory)
        with open(load_dir / "tokenizer_config.json", "r") as f:
            config = json.load(f)
        instance = cls(vocab_size=config["vocab_size"])
        instance._tokenizer = Tokenizer.from_file(str(load_dir / "full_tokenizer.json"))
        instance.endoftext_id = config["endoftext_id"]
        instance.pad_id = config["pad_id"]
        return instance

    def _cache_special_token_ids(self) -> None:
        """Look up and cache the integer IDs of the special tokens."""
        self.endoftext_id = self._tokenizer.token_to_id(ENDOFTEXT_TOKEN)
        self.pad_id       = self._tokenizer.token_to_id(PAD_TOKEN)

    def get_vocab_size(self) -> int:
        return self._tokenizer.get_vocab_size()


print("BPETokenizer class defined.")


BPETokenizer class defined.


### Train the Tokenizer on 80,000 FineWeb-Edu Documents

We stream 80k documents (up from the original 30k) for a richer vocabulary that better covers
educational English text.  The `sample-10BT` split is ~10 billion tokens, so streaming a fraction
of it is very fast and doesn't require downloading the whole dataset.

In [3]:
from datasets import load_dataset

# ---------------------------------------------------------------------------
# Generator that streams raw text from FineWeb-Edu (educational web text).
# We filter out very short documents (<100 chars) which tend to be boilerplate.
# ---------------------------------------------------------------------------
def stream_fineweb_edu(num_docs: int = 80_000):
    """
    Yields up to `num_docs` text strings from FineWeb-Edu (streaming, no full download).
    80k docs takes ~3-5 minutes on Colab and gives excellent vocabulary coverage.
    """
    print(f"Streaming {num_docs:,} documents from FineWeb-Edu (sample-10BT)...")
    dataset = load_dataset(
        "HuggingFaceFW/fineweb-edu",
        name="sample-10BT",
        split="train",
        streaming=True,   # <-- key: no full dataset download needed!
    )
    count = 0
    for example in dataset:
        text = example.get("text", "")
        if text and len(text.strip()) > 100:   # skip very short / empty docs
            yield text
            count += 1
            if count >= num_docs:
                break

# ---------------------------------------------------------------------------
# Train the tokenizer
# 80k docs -> strong vocabulary coverage for educational English text.
# Takes roughly 3-5 minutes on Colab T4.
# ---------------------------------------------------------------------------
tokenizer = BPETokenizer(vocab_size=16384)
tokenizer.train(texts=stream_fineweb_edu(num_docs=80_000))
tokenizer.save("artifacts/tokenizer")
print(f"Tokenizer trained and saved!  Vocab size: {tokenizer.get_vocab_size():,}")

# Quick sanity check: encode then decode -> should get the original text back
_sample = "The quick brown fox jumps over the lazy dog. 1234567890"
_round_trip = tokenizer.decode(tokenizer.encode(_sample))
assert _round_trip == _sample, f"Round-trip failed: {repr(_round_trip)}"
print("Round-trip encode/decode: OK")


Streaming 80,000 documents from FineWeb-Edu (sample-10BT)...


README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Tokenizer trained and saved!  Vocab size: 16,384
Round-trip encode/decode: OK


## 3. Streaming Preprocessing — Token Shards

We stream a second (independent) pass over FineWeb-Edu, tokenize every document, and write
the resulting token IDs to two compact binary files:

| File | Split | Size |
|------|-------|------|
| `data/processed/train.bin` | 98% of docs | ~20 M tokens |
| `data/processed/val.bin`   |  2% of docs | ~0.4 M tokens |

**Format:** flat `numpy.uint16` arrays (2 bytes per token).  A 20 M token training set
is only **40 MB on disk** — tiny!  We use `np.memmap` during training so the file is
never fully loaded into RAM.

**Split strategy:** every 50th document goes to validation (= 2% val split).  This is
deterministic and independent of the tokenizer training split.

In [4]:
import numpy as np


def preprocess_fineweb(
    tokenizer,
    target_train_tokens: int = 20_000_000,
    output_dir: str = "data/processed",
):
    """
    Stream FineWeb-Edu, tokenize each document, and write uint16 binary shards.

    Parameters
    ----------
    tokenizer          : trained BPETokenizer instance
    target_train_tokens: stop after collecting this many training tokens
    output_dir         : folder to write train.bin and val.bin
    """
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    # Open binary files for writing (uint16 = 2 bytes per token ID)
    train_bin = open(out_path / "train.bin", "wb")
    val_bin   = open(out_path / "val.bin",   "wb")

    train_buf, val_buf       = [], []     # in-memory buffers (flushed periodically)
    train_tokens, val_tokens = 0, 0
    doc_idx = 0

    dataset = load_dataset(
        "HuggingFaceFW/fineweb-edu",
        name="sample-10BT",
        split="train",
        streaming=True,
    )
    print(f"Preprocessing target: {target_train_tokens:,} training tokens...")

    for example in dataset:
        text = example.get("text", "")
        if not text or len(text.strip()) < 100:
            continue   # skip boilerplate / empty docs

        # Tokenize and append a document separator so the model learns document boundaries
        token_ids = tokenizer.encode(text) + [tokenizer.endoftext_id]

        # 98 % train / 2 % validation split (every 50th doc -> val)
        if doc_idx % 50 == 0:
            val_buf.extend(token_ids)
            val_tokens += len(token_ids)
        else:
            train_buf.extend(token_ids)
            train_tokens += len(token_ids)

        doc_idx += 1

        # Flush buffers to disk every 500k tokens to keep RAM low
        if len(train_buf) >= 500_000:
            np.array(train_buf, dtype=np.uint16).tofile(train_bin)
            train_buf.clear()
        if len(val_buf) >= 100_000:
            np.array(val_buf, dtype=np.uint16).tofile(val_bin)
            val_buf.clear()

        # Progress report every 2000 docs
        if doc_idx % 2000 == 0:
            pct = 100.0 * train_tokens / target_train_tokens
            print(f"  Docs: {doc_idx:,} | Train tokens: {train_tokens:,} / {target_train_tokens:,}  ({pct:.1f}%)")

        if train_tokens >= target_train_tokens:
            break   # we have enough training tokens

    # Flush remaining buffer contents
    if train_buf:
        np.array(train_buf, dtype=np.uint16).tofile(train_bin)
    if val_buf:
        np.array(val_buf, dtype=np.uint16).tofile(val_bin)

    train_bin.close()
    val_bin.close()

    train_mb = (train_tokens * 2) / 1e6
    val_mb   = (val_tokens   * 2) / 1e6
    print(f"\nPreprocessing complete!")
    print(f"  Train: {train_tokens:,} tokens  (~{train_mb:.1f} MB on disk)")
    print(f"  Val  : {val_tokens:,}   tokens  (~{val_mb:.1f} MB on disk)")


# ---------------------------------------------------------------------------
# Collect 20 M training tokens (40 MB binary file).  On Colab T4 this takes
# roughly 10-15 minutes due to streaming + tokenization.
# ---------------------------------------------------------------------------
preprocess_fineweb(tokenizer, target_train_tokens=20_000_000)


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Preprocessing target: 20,000,000 training tokens...
  Docs: 2,000 | Train tokens: 2,291,999 / 20,000,000  (11.5%)
  Docs: 4,000 | Train tokens: 4,393,685 / 20,000,000  (22.0%)
  Docs: 6,000 | Train tokens: 6,765,455 / 20,000,000  (33.8%)
  Docs: 8,000 | Train tokens: 8,749,163 / 20,000,000  (43.7%)
  Docs: 10,000 | Train tokens: 10,925,978 / 20,000,000  (54.6%)
  Docs: 12,000 | Train tokens: 12,971,125 / 20,000,000  (64.9%)
  Docs: 14,000 | Train tokens: 15,055,145 / 20,000,000  (75.3%)
  Docs: 16,000 | Train tokens: 17,302,842 / 20,000,000  (86.5%)
  Docs: 18,000 | Train tokens: 19,455,319 / 20,000,000  (97.3%)

Preprocessing complete!
  Train: 20,002,802 tokens  (~40.0 MB on disk)
  Val  : 376,002   tokens  (~0.8 MB on disk)


## 4. GPT-2 Style Model Architecture (~50M Parameters)

We build a **decoder-only Transformer** (same family as GPT-2) fully from scratch using
raw PyTorch tensors.  No `transformers` library is used.

### Architecture Overview
```
Input token IDs  [B, T]
      ↓  GPTEmbeddings  (token emb + positional emb + dropout)
      ↓  [B, T, C=640]
      ↓  TransformerBlock × 6
         ├─ LayerNorm → CausalSelfAttention (10 heads, head_dim=64) → residual
         └─ LayerNorm → GPTMLP (640→2560→640, GELU) → residual
      ↓  Final LayerNorm
      ↓  LM Head Linear(640, 16384)  -- no weight tying
      ↓  Logits [B, T, V=16384]  →  Cross-Entropy Loss
```

### Why Pre-LayerNorm?
GPT-2 uses **Pre-LN** (`x = x + Attn(LN(x))`) instead of the original Post-LN.
Pre-LN trains more stably at large scales because gradients flow through the residual
stream without passing through normalisation layers.

In [5]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from typing import Optional


# ---------------------------------------------------------------------------
# Model Configuration
# ---------------------------------------------------------------------------
@dataclass
class GPTConfig:
    """
    All hyperparameters that define the model architecture.

    The defaults here give ~50.6 M parameters, optimised for a T4 GPU.
    Increasing embedding_dim or num_layers increases params quadratically.
    """
    vocab_size:     int   = 16384   # number of unique tokens (BPE vocabulary)
    context_length: int   = 256     # maximum number of tokens the model sees at once
    embedding_dim:  int   = 640     # C: vector size for every token (was 512, now 640)
    num_layers:     int   = 6       # N: number of stacked Transformer blocks
    num_heads:      int   = 10      # H: number of attention heads (head_dim = C/H = 64)
    ffn_multiplier: int   = 4       # MLP hidden dim = ffn_multiplier * embedding_dim = 2560
    dropout:        float = 0.1     # dropout probability applied in attention + MLP
    weight_tying:   bool  = False   # if True, share token-embedding & LM-head weights


# ---------------------------------------------------------------------------
# Module 1: Embeddings
# ---------------------------------------------------------------------------
class GPTEmbeddings(nn.Module):
    """
    Combines:
      - Token embedding  : maps each token ID to a C-dimensional vector.
      - Position embedding: maps each position (0..T-1) to a C-dimensional vector.

    The two are ADDED together so the model knows both *what* the token is
    and *where* it appears in the sequence.

    Input:  idx  [B, T]  (integer token IDs)
    Output: x    [B, T, C]
    """
    def __init__(self, vocab_size: int, context_length: int, embedding_dim: int, dropout: float = 0.0):
        super().__init__()
        self.context_length    = context_length
        self.token_embedding   = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(context_length, embedding_dim)
        self.dropout           = nn.Dropout(dropout)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        b, t = idx.shape
        # positions = [0, 1, 2, ..., t-1]  (one integer per token)
        positions = torch.arange(0, t, dtype=torch.long, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        return self.dropout(x)   # shape: [B, T, C]


# ---------------------------------------------------------------------------
# Module 2: Causal Self-Attention
# ---------------------------------------------------------------------------
class CausalSelfAttention(nn.Module):
    """
    Multi-Head Causal Self-Attention.

    "Causal" means each token can only attend to PAST tokens (not future ones).
    This is enforced by the lower-triangular mask filled with -inf before softmax.

    Math per head h:
        Q_h, K_h, V_h = split(x @ W_QKV)          each [B, H, T, head_dim]
        A_h = softmax( Q_h @ K_h.T / sqrt(d) )     [B, H, T, T]  (causal mask applied)
        out_h = A_h @ V_h                            [B, H, T, head_dim]
    Concatenate all heads -> project back to C.

    Input/Output: [B, T, C]
    """
    def __init__(self, embedding_dim: int, num_heads: int, context_length: int, dropout: float = 0.0):
        super().__init__()
        assert embedding_dim % num_heads == 0, "embedding_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim  = embedding_dim // num_heads   # = 640 / 10 = 64

        # Single Linear that computes Q, K, V for ALL heads in one matrix multiply
        # Output size = 3 * C (split into Q/K/V afterwards)
        self.qkv_proj  = nn.Linear(embedding_dim, 3 * embedding_dim, bias=True)
        self.out_proj  = nn.Linear(embedding_dim, embedding_dim, bias=True)
        self.attn_dropout  = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # Causal mask: lower-triangular matrix of ones, upper-triangular = 0 (=masked)
        # Registered as a buffer so it moves to GPU with model.to(device) but is NOT
        # a trainable parameter.
        mask = torch.tril(torch.ones(context_length, context_length)).view(
            1, 1, context_length, context_length
        )
        self.register_buffer("causal_mask", mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        # 1. Compute Q, K, V in one go
        qkv = self.qkv_proj(x)                          # [B, T, 3C]
        q, k, v = qkv.chunk(3, dim=-1)                  # each [B, T, C]

        # 2. Reshape into per-head tensors and move head dim to axis 1
        #    [B, T, C] -> [B, T, H, head_dim] -> [B, H, T, head_dim]
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # 3. Scaled dot-product attention scores
        #    [B, H, T, head_dim] @ [B, H, head_dim, T] = [B, H, T, T]
        att_scores = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))

        # 4. Apply causal mask: positions where mask==0 get -inf so softmax -> ~0
        att_scores = att_scores.masked_fill(
            self.causal_mask[:, :, :T, :T] == 0, float("-inf")
        )

        # 5. Softmax over the last dim (key positions)
        att_weights = F.softmax(att_scores, dim=-1)  # [B, H, T, T]
        att_weights = self.attn_dropout(att_weights)

        # 6. Weighted sum of value vectors
        out = att_weights @ v                         # [B, H, T, head_dim]

        # 7. Merge all heads back into a single C-dim vector per token
        out = out.transpose(1, 2).contiguous().view(B, T, C)  # [B, T, C]

        return self.resid_dropout(self.out_proj(out))


# ---------------------------------------------------------------------------
# Module 3: Feed-Forward Network (MLP)
# ---------------------------------------------------------------------------
class GPTMLP(nn.Module):
    """
    Two-layer feed-forward network with GELU activation.

    Expands the embedding from C -> 4C (hidden), applies GELU non-linearity,
    then projects back down to C.  For C=640 the hidden layer is 2560.

    This is where the model does most of its "memorisation" and "reasoning".

    Input/Output: [B, T, C]
    """
    def __init__(self, embedding_dim: int, ffn_multiplier: int = 4, dropout: float = 0.0):
        super().__init__()
        hidden_dim = embedding_dim * ffn_multiplier   # 640 * 4 = 2560
        self.fc_in  = nn.Linear(embedding_dim, hidden_dim, bias=True)   # up-projection
        self.act    = nn.GELU()                                           # smooth non-linearity
        self.fc_out = nn.Linear(hidden_dim, embedding_dim, bias=True)   # down-projection
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x -> 2560 -> GELU -> 640
        return self.dropout(self.fc_out(self.act(self.fc_in(x))))


# ---------------------------------------------------------------------------
# Module 4: Transformer Block
# ---------------------------------------------------------------------------
class TransformerBlock(nn.Module):
    """
    One full Transformer layer using Pre-LayerNorm style:

        x = x + Attention( LayerNorm(x) )   # attend, then add residual
        x = x + MLP( LayerNorm(x) )         # transform, then add residual

    The residual connections (+) let gradients flow directly from the final
    output all the way back to the first embedding layer, preventing vanishing
    gradients in deep networks.

    Input/Output: [B, T, C]
    """
    def __init__(
        self,
        embedding_dim: int,
        num_heads: int,
        context_length: int,
        ffn_multiplier: int = 4,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.ln_1  = nn.LayerNorm(embedding_dim)   # LayerNorm before attention
        self.attn  = CausalSelfAttention(embedding_dim, num_heads, context_length, dropout)
        self.ln_2  = nn.LayerNorm(embedding_dim)   # LayerNorm before MLP
        self.mlp   = GPTMLP(embedding_dim, ffn_multiplier, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln_1(x))   # attention sub-layer
        x = x + self.mlp(self.ln_2(x))    # MLP sub-layer
        return x


# ---------------------------------------------------------------------------
# Module 5: Full GPT Model
# ---------------------------------------------------------------------------
class GPT(nn.Module):
    """
    Complete GPT-2 style language model.

    forward(idx) -> logits [B, T, V]
    forward(idx, targets) -> (logits, cross_entropy_loss)
    """
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config     = config
        self.embeddings = GPTEmbeddings(
            config.vocab_size, config.context_length, config.embedding_dim, config.dropout
        )
        # Stack N identical Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(
                config.embedding_dim, config.num_heads, config.context_length,
                config.ffn_multiplier, config.dropout
            )
            for _ in range(config.num_layers)
        ])
        # Final layer norm before the LM head (stabilises output distribution)
        self.ln_f    = nn.LayerNorm(config.embedding_dim)
        # LM head: projects C-dim hidden state to vocabulary logits
        self.lm_head = nn.Linear(config.embedding_dim, config.vocab_size, bias=False)

        # Initialise all weights with a small normal distribution (GPT-2 convention)
        self.apply(self._init_weights)

    def _init_weights(self, module: nn.Module) -> None:
        """Standard GPT-2 weight initialisation."""
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.ones_(module.weight)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

    def forward(self, idx: torch.Tensor, targets: Optional[torch.Tensor] = None):
        """
        Parameters
        ----------
        idx     : [B, T]  integer token IDs (input)
        targets : [B, T]  integer token IDs (next-token labels, optional)

        Returns
        -------
        logits : [B, T, V]  raw unnormalised scores for each vocab token
        loss   : scalar cross-entropy loss  (None if targets not provided)
        """
        x      = self.embeddings(idx)      # [B, T, C]
        for block in self.blocks:
            x  = block(x)                  # [B, T, C]
        x      = self.ln_f(x)             # [B, T, C]
        logits = self.lm_head(x)          # [B, T, V]

        loss = None
        if targets is not None:
            # Flatten to [B*T, V] vs [B*T] for cross-entropy
            loss = F.cross_entropy(
                logits.view(-1, self.config.vocab_size),
                targets.view(-1),
            )
        return logits, loss

    def count_parameters(self) -> int:
        """Print and return total trainable parameter count."""
        total = sum(p.numel() for p in self.parameters())
        print(f"Total Parameters: {total:,}  ({total / 1e6:.2f} M)")
        return total


# ---------------------------------------------------------------------------
# Instantiate and count parameters
# ---------------------------------------------------------------------------
config = GPTConfig()
model  = GPT(config)
model.count_parameters()

# Quick shape check
_x = torch.zeros(2, 8, dtype=torch.long)
_logits, _ = model(_x)
assert _logits.shape == (2, 8, config.vocab_size), "Unexpected output shape!"
print(f"Output shape check: {_logits.shape}  -> OK")
del _x, _logits


Total Parameters: 50,677,760  (50.68 M)
Output shape check: torch.Size([2, 8, 16384])  -> OK


## 5. Dataset Loader & Optimizer Setup

### TokenDataset
Reads from the uint16 binary shard using `np.memmap` (memory-mapped file).  The entire file
is never loaded into RAM; the OS pages in only what is needed.  Each call to `__getitem__`
returns a window of length `context_length+1` which is split into `(x, y)` where `y = x[1:]`
(next-token prediction targets).

### Optimizer: AdamW with weight decay separation
We follow the GPT-2 paper: apply L2 weight decay **only** to 2D parameters (weight matrices)
and NOT to biases, LayerNorm scales, or embedding weights.  This prevents over-regularising
the bias terms which don't contribute to overfitting.

### Cosine LR schedule with warm-up
The learning rate:
1. **Linearly increases** from 0 to `learning_rate` over `warmup_steps`.
2. **Cosine-decays** down to `min_lr` over the remaining steps.

This prevents instability at the start and allows the model to fine-tune at the end.

In [6]:
# ---------------------------------------------------------------------------
# Dataset: memory-mapped binary token shard
# ---------------------------------------------------------------------------
class TokenDataset(Dataset):
    """
    PyTorch Dataset that reads token IDs from a binary uint16 file.

    The file is memory-mapped so it stays on disk; only accessed pages are
    loaded into RAM.  This allows training on datasets far larger than VRAM.
    """
    def __init__(self, bin_path: str, context_length: int = 256):
        # np.memmap: file-backed array, mode='r' = read-only
        self.data          = np.memmap(bin_path, dtype=np.uint16, mode="r")
        self.context_length = context_length
        # Number of valid start positions (we need T+1 tokens per example)
        self.n_examples    = max(0, len(self.data) - context_length - 1)

    def __len__(self) -> int:
        return self.n_examples

    def __getitem__(self, idx: int):
        T = self.context_length
        # Read T+1 consecutive tokens, cast to int64 for PyTorch embedding
        chunk = torch.from_numpy(self.data[idx : idx + T + 1].astype(np.int64))
        x = chunk[:-1]   # input:  tokens 0..T-1
        y = chunk[1:]    # target: tokens 1..T  (each token predicts the next)
        return x, y


def infinite_iter(dataloader):
    """Cycle through a DataLoader forever (step-based training instead of epoch-based)."""
    while True:
        for batch in dataloader:
            yield batch


# ---------------------------------------------------------------------------
# AdamW Optimizer with weight-decay separation
# ---------------------------------------------------------------------------
def configure_optimizers(
    model,
    learning_rate: float = 3e-4,
    weight_decay: float  = 0.1,
    betas=(0.9, 0.95),
):
    """
    Build an AdamW optimiser that applies weight decay only to 2-D weight matrices,
    not to biases / LayerNorm parameters (which are 1-D vectors).
    """
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if param.dim() >= 2:
            decay.append(param)      # weight matrices -> apply L2 regularisation
        else:
            no_decay.append(param)   # biases, LN weights -> no regularisation
    param_groups = [
        {"params": decay,    "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]
    return torch.optim.AdamW(param_groups, lr=learning_rate, betas=betas)


# ---------------------------------------------------------------------------
# Cosine LR Scheduler with Linear Warm-Up
# ---------------------------------------------------------------------------
class CosineWarmupScheduler:
    """
    Manual LR scheduler (does not subclass torch LR scheduler).

    Call `scheduler.step(step_num)` once per training step, AFTER the optimizer step.
    It directly mutates the optimizer's `lr` param group.
    """
    def __init__(self, optimizer, learning_rate: float, min_lr: float, warmup_steps: int, max_steps: int):
        self.optimizer      = optimizer
        self.learning_rate  = learning_rate
        self.min_lr         = min_lr
        self.warmup_steps   = warmup_steps
        self.max_steps      = max_steps

    def step(self, step: int) -> float:
        if step < self.warmup_steps:
            # Linear warm-up: LR ramps from ~0 to learning_rate
            lr = self.learning_rate * (step + 1) / max(1, self.warmup_steps)
        elif step > self.max_steps:
            # After training ends keep LR at min
            lr = self.min_lr
        else:
            # Cosine annealing
            progress = (step - self.warmup_steps) / max(1, self.max_steps - self.warmup_steps)
            lr = self.min_lr + 0.5 * (1.0 + math.cos(math.pi * progress)) * (self.learning_rate - self.min_lr)

        for group in self.optimizer.param_groups:
            group["lr"] = lr
        return lr


print("Dataset, Optimizer, and LR Scheduler defined.")


Dataset, Optimizer, and LR Scheduler defined.


## 6. Pre-Training with Automatic Mixed Precision (AMP)

### Key hyperparameters for T4 (16 GB VRAM)
| Hyperparameter | Value | Rationale |
|---|---|---|
| `batch_size` | 32 | fits comfortably in 16 GB with C=640 |
| `grad_accum_steps` | 4 | effective batch = **128** sequences |
| `max_steps` | 5,000 | ~640 M tokens seen; good for a 50M model |
| `warmup_steps` | 500 | 10% of max_steps |
| `eval_interval` | 500 | evaluate every 10% of training |
| `learning_rate` | 3e-4 | AdamW default for GPT-scale models |
| `min_lr` | 3e-5 | 10% of peak LR (cosine decay floor) |
| `grad_clip` | 1.0 | prevents gradient explosion |

### Gradient Accumulation
Instead of one large batch per step, we do `grad_accum_steps` forward/backward passes
with a smaller micro-batch and accumulate gradients before each optimiser update.
This achieves the same effective batch size as `batch_size * grad_accum_steps = 128`
while using much less VRAM.

### AMP (Automatic Mixed Precision)
PyTorch's `autocast` runs the forward pass in **FP16** (half-precision), which roughly
**halves VRAM usage** and speeds up T4 by ~2x (T4 has dedicated Tensor Cores for FP16).
`GradScaler` prevents FP16 gradient underflow by scaling losses before backward.

In [7]:
import time

# ---------------------------------------------------------------------------
# Hardware setup
# ---------------------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device.upper()}")
if device == "cuda":
    print(f"  GPU : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ---------------------------------------------------------------------------
# Model & Optimizer
# ---------------------------------------------------------------------------
model     = GPT(config).to(device)
optimizer = configure_optimizers(model, learning_rate=3e-4, weight_decay=0.1)

# ---------------------------------------------------------------------------
# Training hyperparameters  (tuned for T4 16 GB)
# ---------------------------------------------------------------------------
max_steps        = 5000    # total optimiser steps  (was 1000)
warmup_steps     = 500     # linear warm-up steps  (was 100)
eval_interval    = 500     # run validation every N steps  (was 100)
batch_size       = 32      # micro-batch per forward pass  (was 16)
grad_accum_steps = 4       # accumulate this many micro-batches  (was 2)
# Effective batch size = batch_size * grad_accum_steps = 32 * 4 = 128 sequences
# Tokens per effective batch = 128 * 256 = 32,768 tokens

print(f"\nEffective batch size : {batch_size * grad_accum_steps} sequences")
print(f"Tokens per step      : {batch_size * grad_accum_steps * config.context_length:,}")
print(f"Total steps          : {max_steps:,}")
print(f"Total tokens seen    : ~{batch_size * grad_accum_steps * config.context_length * max_steps / 1e6:.1f} M")

# ---------------------------------------------------------------------------
# LR Scheduler & AMP Scaler
# ---------------------------------------------------------------------------
scheduler = CosineWarmupScheduler(
    optimizer, learning_rate=3e-4, min_lr=3e-5,
    warmup_steps=warmup_steps, max_steps=max_steps
)
# GradScaler is only meaningful on CUDA; on CPU it's a no-op
scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

# ---------------------------------------------------------------------------
# Data loaders
# ---------------------------------------------------------------------------
train_ds  = TokenDataset("data/processed/train.bin", context_length=config.context_length)
val_ds    = TokenDataset("data/processed/val.bin",   context_length=config.context_length)

# num_workers=2 pre-fetches batches on CPU while the GPU runs the forward pass
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  drop_last=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, drop_last=True,  num_workers=2, pin_memory=True)
train_iter   = infinite_iter(train_loader)

print(f"\nTrain examples : {len(train_ds):,}")
print(f"Val   examples : {len(val_ds):,}")

# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------
print("\nStarting pre-training...")
model.train()
t0 = time.time()
best_val_loss = float("inf")
log_interval  = 50    # print training loss every N steps

for step in range(1, max_steps + 1):
    optimizer.zero_grad(set_to_none=True)  # free grad memory before accumulating
    accum_loss = 0.0

    # --- Gradient accumulation loop ---
    for _ in range(grad_accum_steps):
        x, y = next(train_iter)
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        # autocast: forward pass in FP16 for speed/memory savings
        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            _, loss = model(x, y)
            # Divide loss by accum steps so gradients sum to the true mean
            loss = loss / grad_accum_steps

        accum_loss += loss.item() * grad_accum_steps   # track un-divided loss
        scaler.scale(loss).backward()                  # accumulate scaled gradients

    # --- Optimiser step ---
    scaler.unscale_(optimizer)                          # unscale before clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # prevent gradient explosion
    scaler.step(optimizer)
    scaler.update()
    lr = scheduler.step(step)                           # update learning rate

    # --- Logging ---
    if step % log_interval == 0 or step == 1:
        elapsed = time.time() - t0
        # tokens processed in the last log_interval steps
        toks_per_sec = (log_interval * batch_size * grad_accum_steps * config.context_length) / max(1e-5, elapsed)
        print(f"Step {step:>5}/{max_steps} | Loss: {accum_loss:.4f} | LR: {lr:.2e} | {toks_per_sec:,.0f} tok/s")
        t0 = time.time()

    # --- Validation & Checkpointing ---
    if step % eval_interval == 0 or step == max_steps:
        model.eval()
        val_loss, count = 0.0, 0
        with torch.no_grad():
            for i, (vx, vy) in enumerate(val_loader):
                if i >= 50:   # evaluate on 50 batches (~1600 sequences)
                    break
                vx, vy = vx.to(device, non_blocking=True), vy.to(device, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                    _, vloss = model(vx, vy)
                val_loss += vloss.item()
                count    += 1
        val_loss /= max(1, count)
        perplexity = math.exp(min(val_loss, 20.0))   # cap exponent to avoid overflow
        print(f"  --> [Eval @ step {step}] Val Loss: {val_loss:.4f} | Perplexity: {perplexity:.2f}")

        # Save checkpoint whenever validation loss improves
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            os.makedirs("checkpoints", exist_ok=True)
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "config"          : config.__dict__,
                    "val_loss"        : val_loss,
                    "step"            : step,
                },
                "checkpoints/best.pt",
            )
            print(f"     Checkpoint saved  (val_loss={val_loss:.4f})")
        model.train()

print("\nPre-training finished!")
print(f"Best validation loss : {best_val_loss:.4f}")
print(f"Best perplexity      : {math.exp(min(best_val_loss, 20.0)):.2f}")


Training on: CUDA
  GPU : Tesla T4
  VRAM: 15.6 GB

Effective batch size : 128 sequences
Tokens per step      : 32,768
Total steps          : 5,000
Total tokens seen    : ~163.8 M

Train examples : 20,002,545
Val   examples : 375,745

Starting pre-training...


/tmp/ipykernel_600/1029152121.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))
/tmp/ipykernel_600/1029152121.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):


Step     1/5000 | Loss: 39.3207 | LR: 1.20e-06 | 381,705 tok/s
Step    50/5000 | Loss: 33.7175 | LR: 3.06e-05 | 44,798 tok/s
Step   100/5000 | Loss: 30.1355 | LR: 6.06e-05 | 42,258 tok/s
Step   150/5000 | Loss: 28.4202 | LR: 9.06e-05 | 41,320 tok/s
Step   200/5000 | Loss: 26.8668 | LR: 1.21e-04 | 41,780 tok/s
Step   250/5000 | Loss: 26.0715 | LR: 1.51e-04 | 41,445 tok/s
Step   300/5000 | Loss: 25.3167 | LR: 1.81e-04 | 41,511 tok/s
Step   350/5000 | Loss: 24.5815 | LR: 2.11e-04 | 41,492 tok/s
Step   400/5000 | Loss: 24.2891 | LR: 2.41e-04 | 41,445 tok/s
Step   450/5000 | Loss: 23.6989 | LR: 2.71e-04 | 41,482 tok/s
Step   500/5000 | Loss: 23.3661 | LR: 3.00e-04 | 41,454 tok/s


/tmp/ipykernel_600/1029152121.py:109: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):


  --> [Eval @ step 500] Val Loss: 5.8636 | Perplexity: 352.00
     Checkpoint saved  (val_loss=5.8636)
Step   550/5000 | Loss: 23.1086 | LR: 3.00e-04 | 37,697 tok/s
Step   600/5000 | Loss: 22.7260 | LR: 3.00e-04 | 41,497 tok/s
Step   650/5000 | Loss: 22.2033 | LR: 2.99e-04 | 41,478 tok/s
Step   700/5000 | Loss: 21.9786 | LR: 2.99e-04 | 41,476 tok/s
Step   750/5000 | Loss: 21.5438 | LR: 2.98e-04 | 41,488 tok/s
Step   800/5000 | Loss: 21.4034 | LR: 2.97e-04 | 41,499 tok/s
Step   850/5000 | Loss: 20.9629 | LR: 2.96e-04 | 41,501 tok/s
Step   900/5000 | Loss: 20.9465 | LR: 2.95e-04 | 41,477 tok/s
Step   950/5000 | Loss: 20.7758 | LR: 2.93e-04 | 41,527 tok/s
Step  1000/5000 | Loss: 20.3054 | LR: 2.92e-04 | 41,524 tok/s
  --> [Eval @ step 1000] Val Loss: 5.2134 | Perplexity: 183.72
     Checkpoint saved  (val_loss=5.2134)
Step  1050/5000 | Loss: 20.2610 | LR: 2.90e-04 | 37,747 tok/s
Step  1100/5000 | Loss: 20.1725 | LR: 2.88e-04 | 41,536 tok/s
Step  1150/5000 | Loss: 19.9316 | LR: 2.86e-04 | 

## 7. Load Best Checkpoint & Text Generation

Load the best model saved during training and generate text using three sampling strategies:

| Strategy | What it does |
|---|---|
| **Temperature** | Divide logits by T before softmax.  T<1 = more peaked (conservative), T>1 = flatter (creative) |
| **Top-K** | Keep only the K most likely tokens and re-normalise.  Prevents the model from sampling very rare tokens |
| **Top-P (Nucleus)** | Keep the smallest set of tokens whose cumulative probability exceeds P.  Adapts the cut-off dynamically |

In [8]:
# ---------------------------------------------------------------------------
# Load best checkpoint
# ---------------------------------------------------------------------------
checkpoint = torch.load("checkpoints/best.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded checkpoint from step {checkpoint['step']}  (val_loss={checkpoint['val_loss']:.4f})")


# ---------------------------------------------------------------------------
# Generation function with Temperature + Top-K + Top-P sampling
# ---------------------------------------------------------------------------
@torch.no_grad()
def generate_text(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 150,
    temperature: float  = 0.8,
    top_k: int          = 50,
    top_p: float        = 0.9,
    device: str         = "cuda",
) -> str:
    """
    Autoregressively generate `max_new_tokens` tokens given a text prompt.

    At each step:
      1. Run forward pass -> logits for next token
      2. Apply temperature scaling
      3. Apply Top-K filtering
      4. Apply Top-P (nucleus) filtering
      5. Sample one token from the resulting distribution
      6. Append it and repeat
    """
    model.eval()
    prompt_ids = tokenizer.encode(prompt)
    if not prompt_ids:
        prompt_ids = [tokenizer.endoftext_id]   # fallback for empty prompt
    idx = torch.tensor([prompt_ids], dtype=torch.long, device=device)  # [1, T_prompt]

    for _ in range(max_new_tokens):
        # Truncate context to the model's maximum length
        idx_cond = idx if idx.size(1) <= model.config.context_length else idx[:, -model.config.context_length:]

        # Forward pass
        logits, _ = model(idx_cond)          # [1, T, V]
        logits    = logits[:, -1, :]          # only the last token's logits -> [1, V]
        logits    = logits / temperature      # temperature scaling

        # Top-K: zero out all logits below the K-th largest
        if top_k is not None and top_k > 0:
            top_values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < top_values[:, [-1]]] = -float("Inf")

        # Top-P (nucleus sampling): keep only the most probable tokens summing to >= P
        if top_p is not None and 0.0 < top_p < 1.0:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            # Remove tokens once cumulative probability exceeds top_p
            sorted_remove = cum_probs > top_p
            sorted_remove[..., 1:] = sorted_remove[..., :-1].clone()  # shift right by 1
            sorted_remove[..., 0]  = 0   # always keep the most probable token
            # Scatter back to original logit ordering
            remove = sorted_remove.scatter(1, sorted_idx, sorted_remove)
            logits[remove] = -float("Inf")

        # Sample one token from the resulting distribution
        probs      = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)  # [1, 1]
        idx        = torch.cat((idx, next_token), dim=1)       # append to sequence

        # Stop early if the model emits the end-of-text token
        if next_token.item() == tokenizer.endoftext_id:
            break

    return tokenizer.decode(idx[0].tolist())


# ---------------------------------------------------------------------------
# Test generation with a few prompts
# ---------------------------------------------------------------------------
prompts = [
    "Artificial intelligence and machine learning",
    "The solar system consists of",
    "In mathematics, a prime number is",
]

for prompt in prompts:
    print(f"\nPrompt: {prompt}")
    print("-" * 70)
    out = generate_text(model, tokenizer, prompt, max_new_tokens=150, temperature=0.8, top_k=50, top_p=0.9, device=device)
    print(out)
    print("-" * 70)


Loaded checkpoint from step 5000  (val_loss=4.3513)

Prompt: Artificial intelligence and machine learning
----------------------------------------------------------------------
Artificial intelligence and machine learning. The more people learn about AI and AI, the more the more people use the technology they’ve ever heard from.
The “K” AI system, the more people are, the less you know, the more you are. And it will be the more you’re on a better job. And it will become more complicated when you’re in a real sense, and more people become increasingly aware of the dangers of AI, and not just the world. And it will become a part of the world’s history and technology — as well as the world’s population, it will be far more ambitious than ever.
It’s an enormous step toward the first time of a big business. And there is a great opportunity
----------------------------------------------------------------------

Prompt: The solar system consists of
--------------------------------------------

## 8. Publish Model & Tokenizer to Hugging Face Hub

1. Get a **write token** from https://huggingface.co/settings/tokens
2. Run the cell below and follow the login prompt.
3. Enter your desired repo ID (e.g. `your-username/tiny-llm-50m`).

The cell will upload:
- `pytorch_model.bin` — model weights
- `config.json` — architecture hyperparameters
- `full_tokenizer.json` — complete tokenizer state
- `vocab.json` + `merges.txt` — BPE vocabulary & merge rules
- `tokenizer_config.json` — special token IDs
- `README.md` — auto-generated model card

In [10]:
import torch
from pathlib import Path

# Ensure export directories exist
export_dir = Path("export_hf")
export_dir.mkdir(parents=True, exist_ok=True)

# Save the model state dict to standard PyTorch .pt format
torch_save_path = export_dir / "model.pt"
torch.save(model.state_dict(), torch_save_path)

print(f"Successfully saved the model state dict to {torch_save_path}")

Successfully saved the model state dict to export_hf/model.pt
